In [1]:
library('SelectSim')
library('tidyverse')
library('ggplot2')
library('ggpubr')

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
sessionInfo()

R version 4.4.2 (2024-10-31)
Platform: x86_64-conda-linux-gnu
Running under: AlmaLinux 9.3 (Shamrock Pampas Cat)

Matrix products: default
BLAS/LAPACK: /mnt/ndata/arvind/envs/selectsim_R/lib/libopenblasp-r0.3.29.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Zurich
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] ggpubr_0.6.0      lubridate_1.9.4   forcats_1.0.0     stringr_1.5.1    
 [5] dplyr_1.1.4       purrr_1.0.4       readr_2.1.5       tidyr_1.3.1      
 [9] tibble_3.2.1      ggplot2_3.5.1     tidyverse_2.0.0   Sele

In [3]:
`%notin%` <- Negate(`%in%`)

# Metadata 

In [4]:
genie_metadata_15_msk_with_mutation_selected_final_atleast_100_samples<-readRDS(file='/mnt/ndata/arvind/co_mutation_project/data/processed/genie/msk_one_patient_one_sample_final.rds')
genie_metadata_15_dfci_with_mutation_selected_final_atleast_100_samples<-readRDS(file='/mnt/ndata/arvind/co_mutation_project/data/processed/genie/dfci_one_patient_one_sample_final.rds')

# MAF

In [5]:
genie_maf <- read.delim('/mnt/ndata/arvind/co_mutation_project/data/raw/genie/data_mutations_extended.txt')

# MSK GAMs Generation (hotspot)

In [6]:
genie_maf_msk <- genie_maf %>% select(Chromosome,Start_Position,End_Position,Hugo_Symbol,Variant_Classification,Tumor_Sample_Barcode,HGVSp_Short) %>%
filter(Tumor_Sample_Barcode %in% genie_metadata_15_msk_with_mutation_selected_final_atleast_100_samples$SAMPLE_ID)

In [7]:
msk_genes<-oncokb_genes

In [8]:
genie_maf_msk$sample<-genie_maf_msk$Tumor_Sample_Barcode

In [9]:
input_maf <- genie_maf_msk
print(paste('##### Number of lines ####',nrow(input_maf),sep="->"))
genes_to_consider =  msk_genes
print(paste('##### Number of genes ####',length(genes_to_consider),sep="->"))

[1] "##### Number of lines ####->413413"
[1] "##### Number of genes ####->396"


In [10]:
mutation_type = list(
      'ignore' = c("Silent","Intron","RNA","3'UTR","5'UTR","5'Flank","3'Flank","IGR"),
      'truncating'= c('Frame_Shift_Del','Frame_Shift_Ins','In_Frame_Del','In_Frame_Ins','Nonsense_Mutation','Nonstop_Mutation','Splice_Region','Splice_Site','Translation_Start_Site'),
      'missense' = c('Missense_Mutation')
)
custom_maf_schema = list(
    'name' = 'custom_maf',
    'column' = list(
          'gene' = 'Hugo_Symbol'
        , 'gene.name' = 'Hugo_Symbol'
        , 'sample' = 'sample'
        , 'sample.name' = 'sample'
        , 'mutation.type' = 'Variant_Classification'
        , 'mutation' = 'HGVSp_Short'
        ),
        'mutation.type' = mutation_type
)

In [11]:
head(genie_maf_msk)

,Chromosome,Start_Position,End_Position,Hugo_Symbol,Variant_Classification,Tumor_Sample_Barcode,HGVSp_Short,sample
,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,118165602,118165602,FAM46C,Nonsense_Mutation,GENIE-MSK-P-0081662-T01-IM7,p.R38*,GENIE-MSK-P-0081662-T01-IM7
2,9,27209136,27209136,TEK,Missense_Mutation,GENIE-MSK-P-0039091-T01-IM6,p.D865Y,GENIE-MSK-P-0039091-T01-IM6
3,20,31372564,31372564,DNMT3B,Missense_Mutation,GENIE-MSK-P-0013676-T01-IM5,p.D69Y,GENIE-MSK-P-0013676-T01-IM5
4,4,143003288,143003288,INPP4B,Missense_Mutation,GENIE-MSK-P-0043596-T01-IM6,p.K846N,GENIE-MSK-P-0043596-T01-IM6
5,17,70118924,70118926,SOX9,In_Frame_Del,GENIE-MSK-P-0076002-T01-IM7,p.K167del,GENIE-MSK-P-0076002-T01-IM7
6,16,347904,347904,AXIN1,Missense_Mutation,GENIE-MSK-P-0002914-T01-IM3,p.H534Q,GENIE-MSK-P-0002914-T01-IM3


In [12]:
mut_samples = unique(input_maf[, custom_maf_schema$column$sample])
print(paste('##### Number of samples ####',length(mut_samples),sep="->"))

[1] "##### Number of samples ####->42790"


In [13]:
maf_genes = filter_maf_gene.name(input_maf, genes = genes_to_consider, gene.col = custom_maf_schema$column$gene)
print(paste('##### Number of lines ####',nrow(maf_genes),sep="->"))

[1] "##### Number of lines ####->336381"


In [14]:
head(input_maf)

,Chromosome,Start_Position,End_Position,Hugo_Symbol,Variant_Classification,Tumor_Sample_Barcode,HGVSp_Short,sample
,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
1,1,118165602,118165602,FAM46C,Nonsense_Mutation,GENIE-MSK-P-0081662-T01-IM7,p.R38*,GENIE-MSK-P-0081662-T01-IM7
2,9,27209136,27209136,TEK,Missense_Mutation,GENIE-MSK-P-0039091-T01-IM6,p.D865Y,GENIE-MSK-P-0039091-T01-IM6
3,20,31372564,31372564,DNMT3B,Missense_Mutation,GENIE-MSK-P-0013676-T01-IM5,p.D69Y,GENIE-MSK-P-0013676-T01-IM5
4,4,143003288,143003288,INPP4B,Missense_Mutation,GENIE-MSK-P-0043596-T01-IM6,p.K846N,GENIE-MSK-P-0043596-T01-IM6
5,17,70118924,70118926,SOX9,In_Frame_Del,GENIE-MSK-P-0076002-T01-IM7,p.K167del,GENIE-MSK-P-0076002-T01-IM7
6,16,347904,347904,AXIN1,Missense_Mutation,GENIE-MSK-P-0002914-T01-IM3,p.H534Q,GENIE-MSK-P-0002914-T01-IM3


In [15]:
genie_maf_msk %>% count(Variant_Classification)

Variant_Classification,n
<chr>,<int>
3'Flank,878
3'UTR,122
5'Flank,4774
5'UTR,253
Frame_Shift_Del,39357
Frame_Shift_Ins,14971
In_Frame_Del,6858
In_Frame_Ins,1483
Intron,725


In [16]:
tictoc::tic('##### Creating Truncating GAM ####')
    maf_trunc = filter_maf_truncating(maf_genes,genes=oncokb_truncating_genes, custom_maf_schema)
    input_maf_trunc<-filter_maf_truncating(input_maf, custom_maf_schema)
    truncating_tmb <- data.frame('sample'=mut_samples,'mutation'=rep(0,length(mut_samples)))
    rownames(truncating_tmb)<-mut_samples
    temp <- input_maf_trunc %>% count(sample) 
    rownames(temp)<-temp$sample
    truncating_tmb[intersect(truncating_tmb$sample,temp$sample),]$mutation <-temp[intersect(truncating_tmb$sample,temp$sample),'n']
    tcga_truc_gam = maf2gam(maf_trunc,
                     sample.col = custom_maf_schema$column$sample,
                     gene.col = custom_maf_schema$column$gene,
                     value.var = 'Variant_Classification',
                     samples = mut_samples,
                     genes = genes_to_consider,
                     fun.aggregate = length,
                     binarize=TRUE,
                     fill=0)
    truncating_data <- list('gam'=tcga_truc_gam,
                            'tmb'=truncating_tmb)
tictoc::toc()

##### Creating Truncating GAM ####: 14.353 sec elapsed


In [17]:
#tictoc::tic('##### Creating Missense GAM ####')
    maf_valid = filter_maf_schema(input_maf,
                             schema = custom_maf_schema,
                             column = 'mutation.type',
                             values = custom_maf_schema[['mutation.type']][['ignore']],
                             inclusive = FALSE)
    missense_maf<-filter_maf_mutation.type(input_maf,
                                      variants = 'Missense_Mutation',
                                      variant.col = custom_maf_schema$column$mutation.type)
    missense_tmb <- data.frame('sample'=mut_samples,'mutation'=rep(0,length(mut_samples)))
    rownames(missense_tmb)<-mut_samples
    temp <- missense_maf %>% count(sample) 
    rownames(temp)<-temp$sample
    missense_tmb[intersect(missense_tmb$sample,temp$sample),]$mutation <-temp[intersect(missense_tmb$sample,temp$sample),'n']
    t_m = substr(maf_valid[[custom_maf_schema$column$mutation]],3,1000)
    t_m1 =  gsub('[A-Z]*$', '', t_m)
    maf_valid$HGVSp_Short_fixed = t_m1
    maf_hotspot = filter_maf_mutations(maf_valid,
                                  variant_catalogue,
                                  maf.col = c(custom_maf_schema$column$gene, 'HGVSp_Short_fixed'),
                                  values.col = c('gene', 'mut'))
#tictoc::toc()

In [18]:
head(maf_hotspot,2)

,Hugo_Symbol,HGVSp_Short_fixed,Chromosome,Start_Position,End_Position,Variant_Classification,Tumor_Sample_Barcode,HGVSp_Short,sample,oncogenic
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ABL1,D276,9,133747519,133747519,Missense_Mutation,GENIE-MSK-P-0025939-T01-IH3,p.D276N,GENIE-MSK-P-0025939-T01-IH3,Oncogenic
2,ABL1,E255,9,133738365,133738365,Missense_Mutation,GENIE-MSK-P-0019244-T01-IM6,p.E255D,GENIE-MSK-P-0019244-T01-IM6,Resistance


In [19]:
maf_hotspot_new<-maf_hotspot %>% separate(HGVSp_Short_fixed,remove=FALSE,c('Amino','Position'),sep=1) %>% 
mutate(Hugo_Symbol_new=case_when((Hugo_Symbol=='KRAS'& Amino=='G'& Position==12) ~ "KRAS_G12",
                                 (Hugo_Symbol=='KRAS'& Amino=='G' & Position==13) ~ "KRAS_G13",
                                 (Hugo_Symbol=='KRAS'& Amino=='Q' & Position==61) ~ "KRAS_Q61",
                                 (Hugo_Symbol=='BRAF' & Position==600) ~ "BRAF_1",
                                 (Hugo_Symbol=='BRAF' & HGVSp_Short_fixed %in% c('R462','I463','G464','G469','E586','F595','L597','A598','T598','K601','A727')) ~ "BRAF_2",
                                 (Hugo_Symbol=='BRAF' & HGVSp_Short_fixed %in% c('K484','N581','D594','G596','G466','S467','G469')) ~ "BRAF_3",
                                 (Hugo_Symbol=='PIK3CA' & Position %in% c(32:118)) ~ "PIK3CA_1",
                                 (Hugo_Symbol=='PIK3CA' & Position %in% c(341:483)) ~ "PIK3CA_2",
                                 (Hugo_Symbol=='PIK3CA' & Position %in% c(520:703)) ~ "PIK3CA_3",
                                 (Hugo_Symbol=='PIK3CA' & Position %in% c(798:1014)) ~ "PIK3CA_4",
                                 (Hugo_Symbol=='PIK3CA' & Position %in% c(1015:1068)) ~ "PIK3CA_5",
                                 (Hugo_Symbol=='EGFR' & Position %in% c(185:338)) ~ "EGFR_1",
                                 (Hugo_Symbol=='EGFR' & Position %in% c(713:790)) ~ "EGFR_2",
                                 (Hugo_Symbol=='EGFR' & Position %in% c(800:900)) ~ "EGFR_3",
                                 TRUE ~ as.character(paste(Hugo_Symbol,HGVSp_Short_fixed,sep="_"))))

In [20]:
head(maf_hotspot_new,2)

,Hugo_Symbol,HGVSp_Short_fixed,Amino,Position,Chromosome,Start_Position,End_Position,Variant_Classification,Tumor_Sample_Barcode,HGVSp_Short,sample,oncogenic,Hugo_Symbol_new
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ABL1,D276,D,276,9,133747519,133747519,Missense_Mutation,GENIE-MSK-P-0025939-T01-IH3,p.D276N,GENIE-MSK-P-0025939-T01-IH3,Oncogenic,ABL1_D276
2,ABL1,E255,E,255,9,133738365,133738365,Missense_Mutation,GENIE-MSK-P-0019244-T01-IM6,p.E255D,GENIE-MSK-P-0019244-T01-IM6,Resistance,ABL1_E255


In [21]:
maf_hotspot_new<-maf_hotspot_new %>% 
                 mutate(New_Gene=case_when((Hugo_Symbol %in% c('KRAS','EGFR','PIK3CA','BRAF'))~Hugo_Symbol_new,
                                           TRUE ~ as.character(Hugo_Symbol)))

In [22]:
dim(maf_hotspot_new)
head(maf_hotspot_new,2)

[1] 73799    14

,Hugo_Symbol,HGVSp_Short_fixed,Amino,Position,Chromosome,Start_Position,End_Position,Variant_Classification,Tumor_Sample_Barcode,HGVSp_Short,sample,oncogenic,Hugo_Symbol_new,New_Gene
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ABL1,D276,D,276,9,133747519,133747519,Missense_Mutation,GENIE-MSK-P-0025939-T01-IH3,p.D276N,GENIE-MSK-P-0025939-T01-IH3,Oncogenic,ABL1_D276,ABL1
2,ABL1,E255,E,255,9,133738365,133738365,Missense_Mutation,GENIE-MSK-P-0019244-T01-IM6,p.E255D,GENIE-MSK-P-0019244-T01-IM6,Resistance,ABL1_E255,ABL1


In [23]:
hot_spot_genes<-setdiff(unique(maf_hotspot_new$New_Gene),oncokb_genes)

In [24]:
hot_spot_genes

[1] "BRAF_A246"   "BRAF_2"      "BRAF_D287"   "BRAF_3"      "BRAF_E275"  
 [6] "BRAF_F247"   "BRAF_F468"   "BRAF_H574"   "BRAF_K483"   "BRAF_K499"  
[11] "BRAF_L485"   "BRAF_L505"   "BRAF_P367"   "BRAF_Q257"   "BRAF_R671"  
[16] "BRAF_S151"   "BRAF_T241"   "BRAF_T599"   "BRAF_V471"   "BRAF_1"     
[21] "BRAF_Y472"   "EGFR_1"      "EGFR_2"      "EGFR_3"      "EGFR_C620"  
[26] "EGFR_C797"   "EGFR_D587"   "EGFR_E114"   "EGFR_E709"   "EGFR_E931"  
[31] "EGFR_G465"   "EGFR_G588"   "EGFR_G598"   "EGFR_G983"   "EGFR_I491"  
[36] "EGFR_L62"    "EGFR_P589"   "EGFR_P596"   "EGFR_R108"   "EGFR_R669"  
[41] "EGFR_S492"   "EGFR_S645"   "EGFR_T1041"  "EGFR_T363"   "EGFR_V148"  
[46] "KRAS_A146"   "KRAS_A18"    "KRAS_A59"    "KRAS_D119"   "KRAS_D33"   
[51] "KRAS_E31"    "KRAS_E63"    "KRAS_F28"    "KRAS_G12"    "KRAS_G13"   
[56] "KRAS_G60"    "KRAS_K117"   "KRAS_K147"   "KRAS_K5"     "KRAS_L19"   
[61] "KRAS_P34"    "KRAS_Q22"    "KRAS_Q61"    "KRAS_R149"   "KRAS_S65"   
[66] "KRAS_T58"    "KRAS_T74"    "KRAS_V14"    "KRAS_Y64"    "KRAS_Y71"   
[71] "PIK3CA_5"    "PIK3CA_2"    "PIK3CA_3"    "PIK3CA_4"    "PIK3CA_1"   
[76] "PIK3CA_P124"

In [25]:
missense_tcga_gam = maf2gam(maf_hotspot_new,
                 sample.col = custom_maf_schema$column$sample,
                 gene.col = 'New_Gene',
                 value.var = 'Variant_Classification',
                 samples = mut_samples,
                 genes = c(oncokb_genes,hot_spot_genes),
                 fun.aggregate = length,
                 binarize=TRUE,
                 fill=0)
missesne_data <- list('gam'=missense_tcga_gam,
                      'tmb'=missense_tmb)

In [26]:
str(truncating_data)
str(missesne_data)

List of 2
 $ gam: num [1:42790, 1:396] 0 0 0 0 0 0 0 0 0 0 ...
  ..- attr(*, "dimnames")=List of 2
  .. ..$ : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "GENIE-MSK-P-0013676-T01-IM5" "GENIE-MSK-P-0043596-T01-IM6" ...
  .. ..$ : chr [1:396] "AKT1" "ALK" "APC" "AR" ...
 $ tmb:'data.frame':	42790 obs. of  2 variables:
  ..$ sample  : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "GENIE-MSK-P-0013676-T01-IM5" "GENIE-MSK-P-0043596-T01-IM6" ...
  ..$ mutation: num [1:42790] 45 13 42 87 2 0 1 7 31 0 ...
List of 2
 $ gam: num [1:42790, 1:472] 0 0 0 0 0 0 0 0 0 0 ...
  ..- attr(*, "dimnames")=List of 2
  .. ..$ : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "GENIE-MSK-P-0013676-T01-IM5" "GENIE-MSK-P-0043596-T01-IM6" ...
  .. ..$ : chr [1:472] "AKT1" "ALK" "APC" "AR" ...
 $ tmb:'data.frame':	42790 obs. of  2 variables:
  ..$ sample  : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "

In [27]:
dim(genie_metadata_15_msk_with_mutation_selected_final_atleast_100_samples)

[1] 42790    22

In [28]:
head(genie_metadata_15_msk_with_mutation_selected_final_atleast_100_samples,2)

,PATIENT_ID,SEX,PRIMARY_RACE,ETHNICITY,CENTER,INT_CONTACT,INT_DOD,YEAR_CONTACT,DEAD,YEAR_DEATH,⋯,ONCOTREE_CODE,SAMPLE_TYPE,SEQ_ASSAY_ID,CANCER_TYPE,CANCER_TYPE_DETAILED,SAMPLE_TYPE_DETAILED,class,Tumor_run_group,final_class,final_run_group
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,GENIE-MSK-P-0000223,Female,White,Non-Spanish/non-Hispanic,MSK,11142,11142,2016,TRUE,2016,⋯,AASTR,Primary,MSK-IMPACT341,Glioma,Anaplastic Astrocytoma,Primary tumor,AASTR,AASTR,Glioma_low_grade,Glioma
2,GENIE-MSK-P-0000679,Female,White,Non-Spanish/non-Hispanic,MSK,22267,22267,2015,TRUE,2015,⋯,AASTR,Primary,MSK-IMPACT341,Glioma,Anaplastic Astrocytoma,Primary tumor,AASTR,AASTR,Glioma_low_grade,Glioma


In [29]:
sample_annoation <- (genie_metadata_15_msk_with_mutation_selected_final_atleast_100_samples %>% filter(SAMPLE_ID %in% mut_samples))$final_class
names(sample_annoation)<-(genie_metadata_15_msk_with_mutation_selected_final_atleast_100_samples %>% filter(SAMPLE_ID %in% mut_samples))$SAMPLE_ID

In [30]:
str(sample_annoation)

 Named chr [1:42790] "Glioma_low_grade" "Glioma_low_grade" ...
 - attr(*, "names")= chr [1:42790] "GENIE-MSK-P-0000223-T01-IM3" "GENIE-MSK-P-0000679-T01-IM3" "GENIE-MSK-P-0000748-T01-IM3" "GENIE-MSK-P-0001408-T01-IM3" ...


In [31]:
maf_trunc %>% filter(Hugo_Symbol=='KRAS')

Chromosome,Start_Position,End_Position,Hugo_Symbol,Variant_Classification,Tumor_Sample_Barcode,HGVSp_Short,sample
<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
12,25380276,25380277,KRAS,In_Frame_Ins,GENIE-MSK-P-0049113-T01-IM6,p.L52_G60dup,GENIE-MSK-P-0049113-T01-IM6
12,25398283,25398284,KRAS,In_Frame_Ins,GENIE-MSK-P-0041309-T01-IM6,p.A11_G12dup,GENIE-MSK-P-0041309-T01-IM6
12,25378666,25378669,KRAS,Frame_Shift_Del,GENIE-MSK-P-0008466-T01-IM5,p.P110Lfs*4,GENIE-MSK-P-0008466-T01-IM5
12,25380245,25380245,KRAS,Nonsense_Mutation,GENIE-MSK-P-0053680-T01-IM6,p.Y71*,GENIE-MSK-P-0053680-T01-IM6
12,25368433,25368434,KRAS,Frame_Shift_Ins,GENIE-MSK-P-0007997-T01-IM5,p.I171Nfs*14,GENIE-MSK-P-0007997-T01-IM5
12,25368420,25368422,KRAS,In_Frame_Del,GENIE-MSK-P-0028891-T01-IM6,p.E175del,GENIE-MSK-P-0028891-T01-IM6
12,25398280,25398281,KRAS,In_Frame_Ins,GENIE-MSK-P-0023599-T01-IM6,p.G13dup,GENIE-MSK-P-0023599-T01-IM6
12,25368455,25368455,KRAS,Nonsense_Mutation,GENIE-MSK-P-0047901-T01-IM6,p.R164*,GENIE-MSK-P-0047901-T01-IM6
12,25398287,25398288,KRAS,In_Frame_Ins,GENIE-MSK-P-0044348-T01-IM6,p.G10dup,GENIE-MSK-P-0044348-T01-IM6


## Creating Primary Run GAM

In [32]:
primary_samples<-(genie_metadata_15_msk_with_mutation_selected_final_atleast_100_samples %>% filter(SAMPLE_TYPE=='Primary'))$SAMPLE_ID

In [33]:
gene_to_take <- colnames(missesne_data$gam)
order <- primary_samples

In [34]:
consider_hotspot_genes <- (data.frame('freq'=colSums(missense_tcga_gam[order,]),'gene'=gene_to_take) %>% filter(gene %in% hot_spot_genes) %>% filter(freq>=20))$gene

In [35]:
length(gene_to_take)

[1] 472

In [42]:
length(consider_hotspot_genes)

[1] 19

In [36]:
to_consider_gene<-(unique(setdiff(c(oncokb_genes,consider_hotspot_genes),c('KRAS','BRAF','PIK3CA','EGFR'))))

In [43]:
length(to_consider_gene)

[1] 411

In [44]:
tcga_truc_gam = maf2gam(maf_trunc,
                 sample.col = custom_maf_schema$column$sample,
                 gene.col = custom_maf_schema$column$gene,
                 value.var = 'Variant_Classification',
                 samples = mut_samples,
                 genes = to_consider_gene,
                 fun.aggregate = length,
                 binarize=TRUE,
                 fill=0)
truncating_data <- list('gam'=tcga_truc_gam,
                        'tmb'=truncating_tmb)

In [45]:
str(truncating_data)
str(missesne_data)

List of 2
 $ gam: num [1:42790, 1:411] 0 0 0 0 0 0 0 0 0 0 ...
  ..- attr(*, "dimnames")=List of 2
  .. ..$ : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "GENIE-MSK-P-0013676-T01-IM5" "GENIE-MSK-P-0043596-T01-IM6" ...
  .. ..$ : chr [1:411] "AKT1" "ALK" "APC" "AR" ...
 $ tmb:'data.frame':	42790 obs. of  2 variables:
  ..$ sample  : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "GENIE-MSK-P-0013676-T01-IM5" "GENIE-MSK-P-0043596-T01-IM6" ...
  ..$ mutation: num [1:42790] 45 13 42 87 2 0 1 7 31 0 ...
List of 2
 $ gam: num [1:42790, 1:472] 0 0 0 0 0 0 0 0 0 0 ...
  ..- attr(*, "dimnames")=List of 2
  .. ..$ : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "GENIE-MSK-P-0013676-T01-IM5" "GENIE-MSK-P-0043596-T01-IM6" ...
  .. ..$ : chr [1:472] "AKT1" "ALK" "APC" "AR" ...
 $ tmb:'data.frame':	42790 obs. of  2 variables:
  ..$ sample  : chr [1:42790] "GENIE-MSK-P-0081662-T01-IM7" "GENIE-MSK-P-0039091-T01-IM6" "

In [46]:
# generating only missesne data
gene_to_take <- to_consider_gene
data <-list('M'=list('missense'=t(missesne_data$gam[order,gene_to_take]),
                     'truncating'=t(truncating_data$gam[order,gene_to_take])
                    ),
            'tmb'=list('missense'=missesne_data$tmb[order,],
                      'truncating'=truncating_data$tmb[order,]))

alteration_covariates <- rep('MUT',ncol(missesne_data$gam[order,gene_to_take]))
names(alteration_covariates)<-colnames(missesne_data$gam[order,gene_to_take])

In [47]:
run_data <- list('M'=data,'sample.class' = sample_annoation[order],'alteration.class' = alteration_covariates)
str(run_data)

List of 3
 $ M               :List of 2
  ..$ M  :List of 2
  .. ..$ missense  : num [1:411, 1:27825] 0 0 0 0 0 0 0 0 0 0 ...
  .. .. ..- attr(*, "dimnames")=List of 2
  .. .. .. ..$ : chr [1:411] "AKT1" "ALK" "APC" "AR" ...
  .. .. .. ..$ : chr [1:27825] "GENIE-MSK-P-0000223-T01-IM3" "GENIE-MSK-P-0000679-T01-IM3" "GENIE-MSK-P-0000748-T01-IM3" "GENIE-MSK-P-0001408-T01-IM3" ...
  .. ..$ truncating: num [1:411, 1:27825] 0 0 0 0 0 0 0 0 0 0 ...
  .. .. ..- attr(*, "dimnames")=List of 2
  .. .. .. ..$ : chr [1:411] "AKT1" "ALK" "APC" "AR" ...
  .. .. .. ..$ : chr [1:27825] "GENIE-MSK-P-0000223-T01-IM3" "GENIE-MSK-P-0000679-T01-IM3" "GENIE-MSK-P-0000748-T01-IM3" "GENIE-MSK-P-0001408-T01-IM3" ...
  ..$ tmb:List of 2
  .. ..$ missense  :'data.frame':	27825 obs. of  2 variables:
  .. .. ..$ sample  : chr [1:27825] "GENIE-MSK-P-0000223-T01-IM3" "GENIE-MSK-P-0000679-T01-IM3" "GENIE-MSK-P-0000748-T01-IM3" "GENIE-MSK-P-0001408-T01-IM3" ...
  .. .. ..$ mutation: num [1:27825] 4 4 1 0 4 2 3 3 2 3 ..

In [48]:
saveRDS(run_data,file='../data/pan_can_msk_primary_run_hotspot_v15.rds')

In [49]:
consider_hotspot_genes

[1] "BRAF_2"    "BRAF_3"    "BRAF_1"    "EGFR_1"    "EGFR_2"    "EGFR_3"   
 [7] "EGFR_G598" "EGFR_R108" "KRAS_A146" "KRAS_A59"  "KRAS_G12"  "KRAS_G13" 
[13] "KRAS_K117" "KRAS_Q61"  "PIK3CA_5"  "PIK3CA_2"  "PIK3CA_3"  "PIK3CA_4" 
[19] "PIK3CA_1"

In [50]:
saveRDS(consider_hotspot_genes,file='../data/pan_can_msk_hotspot_genes.rds')